# Entrenamiento desde cero con cortes sin pelvis (negativos)

**Por qué.** El profesor va a probar el sistema con un estudio que puede tener pelvis o no. El modelo original se entrenó solo con cortes con pelvis y, en un volumen completo, marca fémures y columna como si fueran pelvis. El ajuste fino con negativos (`Negativos_ajuste.ipynb`) redujo esos falsos positivos, pero solo en 4 épocas al final. Aquí se repite el entrenamiento **completo** (etapas A, B y C) con los negativos desde la primera época.

**Qué cambia frente al entrenamiento original.** Solo los datos de entrenamiento: los 18.072 cortes con pelvis **más los 5.698 cortes sin un solo vóxel de pelvis** de `train`. Las etapas, los hiperparámetros, la semilla, la entrada 2.5D y los centros "interior" se leen de `resultados/metricas/entrenamiento_multitarea/config.json`, así que son idénticos. La validación de mAP y Dice usa los mismos cortes positivos de siempre; además se miden los cortes vacíos.

**Qué no hace este notebook.** No reemplaza ningún modelo. Deja `modelo_multitarea_negativos_completo.pt` para la comparación final en 3D (celda de comparación en `salida_3d.ipynb`), que aplica la regla acordada:

> **Candidatos:** modelo ajustado con negativos y modelo entrenado desde cero con negativos. El original es la referencia.
> Un candidato **es aceptable** si: (1) frente al original no baja más de 0,01 en mAP@0.50 ni en Dice y conserva al menos el 99 % de los cortes con pelvis; (2) su F1 3D, con la salida 3D reajustada para él, no queda más de un fragmento (0,04) por debajo del original; (3) el sistema completo (modelo + control de "pelvis presente") entrega 0 fragmentos en los 10 tramos sin pelvis de validación.
> Entre los aceptables gana el de mayor F1 3D; si la diferencia es de un fragmento o menos, gana el que invente menos fragmentos sin ayuda del control. Si ninguno es aceptable, se lleva la tabla al equipo.

Este notebook revisa la condición (1) y deja todo listo para (2) y (3).

**Cómo correrlo**
1. `SMOKE_TEST = True` y *Run All*: unos minutos, con 64 cortes y 1 época por etapa. No toca los resultados oficiales.
2. `SMOKE_TEST = False`, reiniciar el kernel y *Run All*: unas 2 horas en la RTX 5060. Las cachés de negativos se reutilizan si ya existen.

**Salidas:** `resultados/metricas/entrenamiento_negativos_completo/`, `resultados/figuras/entrenamiento_negativos_completo/` y `resultados/modelos/entrenamiento_negativos_completo/`.

## 1. Configuración

Todo se lee del `config.json` del entrenamiento original. Lo único propio de este notebook es la fracción de negativos (todos) y las carpetas de salida.

In [ ]:
import ast, gc, hashlib, json, math, random, shutil, subprocess, sys, time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy.ndimage import maximum_filter
from skimage.exposure import equalize_adapthist
from skimage.filters import threshold_otsu

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.txt").is_file())
BASE_DIR = ROOT                                   # lo usan resolve_repo_path y las definiciones del equipo
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.rutas import CACHE_DIR, DATASET_CSV, DATOS_DIR, FIGURAS_DIR, MANIFEST_CSV, METRICAS_DIR, MODELOS_DIR, NOTEBOOKS_DIR
from src.definiciones_notebooks import cargar_definiciones
from src import cortes_vacios as cv

SMOKE_TEST = True          # 1) True: prueba corta que no toca los resultados oficiales. 2) False: corrida real.
SEED = 42

# ---------------------------------------------------------------- modelos de referencia (no se modifican)
ORIGEN_DIR = MODELOS_DIR / "entrenamiento_multitarea"
_respaldo = ORIGEN_DIR / "modelo_multitarea_final_sin_negativos.pt"
MODELO_ORIGINAL = _respaldo if _respaldo.is_file() else ORIGEN_DIR / "modelo_multitarea_final.pt"   # referencia sin negativos
MODELO_AJUSTADO = MODELOS_DIR / "entrenamiento_negativos" / "modelo_multitarea_negativos.pt"         # candidato 1
for _ruta in (MODELO_ORIGINAL, MODELO_AJUSTADO):
    if not _ruta.is_file():
        raise FileNotFoundError(f"No encuentro {_ruta}")
    if _ruta.stat().st_size < 10_000:
        raise RuntimeError(f"{_ruta.name} es un puntero de Git LFS: ejecutar `git lfs install` y `git lfs pull`.")

# ---------------------------------------------------------------- mismos hiperparámetros del entrenamiento original
CONFIG_ORIGINAL = json.loads((METRICAS_DIR / "entrenamiento_multitarea" / "config.json").read_text(encoding="utf-8"))
IMAGE_SIZE = CONFIG_ORIGINAL["IMAGE_SIZE"]
GRID_SIZE = CONFIG_ORIGINAL["GRID_SIZE"]
BACKBONE_NAME = CONFIG_ORIGINAL["BACKBONE_NAME"]
USE_CBAM = CONFIG_ORIGINAL["USE_CBAM"]
USE_CBAM_GATE = CONFIG_ORIGINAL["USE_CBAM_GATE"]
USE_SKIPS = CONFIG_ORIGINAL["USE_SKIPS"]
BATCH_SIZE = CONFIG_ORIGINAL["BATCH_SIZE"]
VAL_BATCH_SIZE = CONFIG_ORIGINAL["VAL_BATCH_SIZE"]
WEIGHT_DECAY = CONFIG_ORIGINAL["WEIGHT_DECAY"]
GRAD_CLIP = CONFIG_ORIGINAL["GRAD_CLIP"]
SEG_WEIGHTS = tuple(CONFIG_ORIGINAL["SEG_WEIGHTS"])
JITTER_PX = CONFIG_ORIGINAL["JITTER_PX"]
PRED_BOX_SCORE_MIN = CONFIG_ORIGINAL["PRED_BOX_SCORE_MIN"]
INSTANCE_EVAL_N = CONFIG_ORIGINAL["INSTANCE_EVAL_N"]
EVAL_SCORE_MIN = CONFIG_ORIGINAL["EVAL_SCORE_MIN"]
IN_CHANNELS = CONFIG_ORIGINAL.get("IN_CHANNELS", 1)          # 3 = entrada 2.5D (cortes z-1, z, z+1)
CENTER_MODE = CONFIG_ORIGINAL.get("CENTER_MODE", "mean")     # centro objetivo del decoder
STAGES = json.loads(json.dumps(CONFIG_ORIGINAL["STAGES"]))   # etapas A, B y C idénticas al original (copia)
NUM_WORKERS = 0
LOG_GRAD_EVERY = 50
MAX_TRAIN_SLICES = MAX_VAL_SLICES = MAX_NEG_SLICES = None

# ---------------------------------------------------------------- lo único nuevo
FRACCION_NEGATIVOS = 1.0           # todos los cortes sin pelvis de train (~1 por cada 3 positivos, la proporción de un volumen)
RECALL_MIN = cv.CRITERIO["recall_min"]
UMBRAL_ACTUAL = json.loads((METRICAS_DIR / "nms_pertenencia" / "umbral_corte.json").read_text(encoding="utf-8"))["umbral_corte"]

if SMOKE_TEST:
    MAX_TRAIN_SLICES, MAX_VAL_SLICES, MAX_NEG_SLICES, INSTANCE_EVAL_N = 64, 32, 32, 8
    BATCH_SIZE = VAL_BATCH_SIZE = 8
    for _st in STAGES:
        _st["epochs"] = 1

ETAPA = "entrenamiento_negativos_completo_smoke" if SMOKE_TEST else "entrenamiento_negativos_completo"
OUT_ROOT, MODEL_ROOT, FIG_DIR = METRICAS_DIR / ETAPA, MODELOS_DIR / ETAPA, FIGURAS_DIR / ETAPA
for _d in (OUT_ROOT, MODEL_ROOT, FIG_DIR, CACHE_DIR):
    _d.mkdir(parents=True, exist_ok=True)
OUT_DIR, MODEL_DIR = OUT_ROOT, MODEL_ROOT          # globals que usa train_stage

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
USE_AMP = DEVICE.type == "cuda"
NOMBRE_DISPOSITIVO = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else DEVICE.type

print(f"PyTorch {torch.__version__} | dispositivo: {NOMBRE_DISPOSITIVO} | AMP: {USE_AMP} | SMOKE_TEST: {SMOKE_TEST}")
print(f"Etapas: {[(s['name'], s['epochs']) for s in STAGES]} | canales de entrada: {IN_CHANNELS} | centros: {CENTER_MODE}")
print("Referencias:", MODELO_ORIGINAL.name, "(original) y", MODELO_AJUSTADO.name, "(ajustado) | umbral_corte actual:", UMBRAL_ACTUAL)
print("Salidas en:", OUT_ROOT)

## 2. Cargar las definiciones del equipo

Se reutiliza el código existente sin copiarlo: datos y targets, backbone, CBAM, decoder y, del notebook `Entrenamiento_multitarea`, el modelo, el caché, las pérdidas, la evaluación y el bucle `train_stage`. Si alguna definición cambia de nombre, la carga se detiene con un error.

In [ ]:
n_def = 0
n_def += cargar_definiciones(NOTEBOOKS_DIR / "Dataset_splits.ipynb", {"resolve_repo_path"}, destino=globals())
n_def += cargar_definiciones(NOTEBOOKS_DIR / "Ventaneo_hu.ipynb", {"apply_hu_window", "enhance_bone_contrast"}, destino=globals())
n_def += cargar_definiciones(NOTEBOOKS_DIR / "Targets_deteccion.ipynb",
                             {"instance_to_semantic", "boxes_from_semantic", "letterbox_pair", "build_detection_target",
                              "PengwinDetectionDataset"}, valores={"CLASS_ID_TO_NAME"}, destino=globals())
n_def += cargar_definiciones(NOTEBOOKS_DIR / "Backbone.ipynb", {"FundidoraPC", "FundidoraPCPlus", "build_backbone"}, destino=globals())
n_def += cargar_definiciones(NOTEBOOKS_DIR / "Segmentacion_instancias.ipynb",
                             {"build_instance_segmentation_targets", "build_detection_region_priors", "SegmentationConvBlock",
                              "backbone_feature_pyramid", "SkipFusionBlock", "CenterOffsetSkipDecoder",
                              "center_offset_segmentation_loss", "soft_dice_semantic_loss",
                              "decode_center_offset_instances"}, destino=globals())
n_def += cargar_definiciones(NOTEBOOKS_DIR / "cbam.ipynb", {"ChannelAttention", "SpatialAttention", "CBAM", "CompuertaResidualCBAM"},
                             destino=globals())
n_def += cargar_definiciones(
    NOTEBOOKS_DIR / "Entrenamiento_multitarea.ipynb",
    {"seed_everything", "autocast", "_sha1", "_VolumeSlices", "LeanPengwinDetectionDataset", "build_or_load_cache", "_read_cache",
     "pad_boxes", "unpad_boxes", "CachedSliceDataset", "collate_slices", "MultiTaskNet", "jitter_boxes", "predicted_boxes_top1",
     "make_priors", "make_grid_targets", "seg_loss_from_batch", "compute_losses", "_to_np", "_iou_np", "_average_precision",
     "detection_metrics", "fragment_hits", "evaluate", "selection_score", "set_trainable", "set_train_mode", "group_grad_norms",
     "trunk_grad_balance", "save_checkpoint", "load_weights", "train_stage", "export_inference_checkpoint", "load_trained_model"},
    destino=globals())

from src.deteccion_grid import (CH_BOX, CH_CLS, CH_OBJ, CLASS_NAMES, GRID_S, IMG_SIZE, LAMBDAS, STRIDE, box_iou, build_targets,
                                decode_boxes, decode_predictions, encode_boxes, grid_detection_loss, GridDetectionHead,
                                nms_por_clase, pairwise_iou, postprocess, split_head_output)
from src.pipeline_pengwin import CONFIG_PIPELINE, scores_por_corte
from src.entrada_25d import apilar_cortes, in_channels_del_modelo

assert GRID_S == GRID_SIZE and IMG_SIZE == IMAGE_SIZE, "la cabeza grid espera otra resolución"
DET_LAMBDAS = dict(LAMBDAS)
assert DET_LAMBDAS == CONFIG_ORIGINAL["DET_LAMBDAS"], "los λ de detección no son los del entrenamiento original"
print(f"Definiciones cargadas: {n_def} | λ detección: {DET_LAMBDAS}")

# Tablas globales que usa PengwinDetectionDataset
df = pd.read_csv(DATASET_CSV, dtype={"case_id": str})
slice_manifest = pd.read_csv(MANIFEST_CSV, dtype={"case_id": str})
DATA_VERSION = {"dataset_csv": _sha1(DATASET_CSV), "manifest": _sha1(MANIFEST_CSV), "image_size": IMAGE_SIZE, "channels": IN_CHANNELS}
if DATA_VERSION != CONFIG_ORIGINAL["DATA_VERSION"]:
    # En Windows Git convierte los CSV a CRLF y la huella cambia aunque los datos sean los mismos: el caché se reconstruye.
    print("AVISO: la huella de dataset.csv / manifiesto no coincide con la del entrenamiento original:",
          DATA_VERSION, "vs", CONFIG_ORIGINAL["DATA_VERSION"], "-> el caché de positivos se reconstruye.")

## 3. Manifiesto de cortes negativos

Un corte es negativo si su label no tiene **ni un vóxel** de pelvis. Los cortes con muy poca pelvis, que tampoco están en el manifiesto de positivos (menos de 20 píxeles), quedan fuera de los dos: son ambiguos y no se usan. Para cada negativo se guarda su distancia a la pelvis, solo como descripción. `posicion` se refiere al índice z del array (`antes` y `despues` del tramo con pelvis), sin suponer qué lado es anatómicamente arriba.

In [ ]:
NEG_CSV = DATOS_DIR / "targets" / "manifest_cortes_negativos.csv"


def leer_label(case_id):
    fila = df[df["case_id"] == case_id].iloc[0]
    return sitk.GetArrayFromImage(sitk.ReadImage(str(resolve_repo_path(fila["label_path"]))))


if NEG_CSV.is_file():
    negativos = pd.read_csv(NEG_CSV, dtype={"case_id": str})
    print("Manifiesto de negativos cargado:", NEG_CSV.relative_to(ROOT))
else:
    t0 = time.time()
    negativos = cv.manifiesto_negativos(df[["case_id", "split"]], leer_label)
    negativos.to_csv(NEG_CSV, index=False)
    print(f"Manifiesto de negativos creado en {time.time() - t0:.0f} s:", NEG_CSV.relative_to(ROOT))

# Ningún corte puede ser positivo y negativo a la vez
cruce = negativos.merge(slice_manifest, on=["case_id", "slice_index"], how="inner")
assert cruce.empty, f"{len(cruce)} cortes están en los dos manifiestos"
# Los splits del manifiesto coinciden con dataset.csv (sin fuga de pacientes)
assert (negativos.merge(df[["case_id", "split"]], on="case_id", suffixes=("", "_ds"))
        .eval("split == split_ds").all()), "el split de algún negativo no coincide con dataset.csv"

resumen_neg = pd.DataFrame({
    "positivos": slice_manifest.groupby("split").size(),
    "negativos": negativos.groupby("split").size(),
}).fillna(0).astype(int)
resumen_neg["negativos_por_positivo"] = (resumen_neg["negativos"] / resumen_neg["positivos"]).round(3)
display(resumen_neg)
print("Posición de los negativos:", negativos["posicion"].value_counts().to_dict())
print("Distancia a la pelvis (cortes):", negativos["distancia_a_pelvis"].describe()[["min", "50%", "max"]].to_dict())

## 4. Cachés de cortes

Los positivos reutilizan el mismo caché del entrenamiento original (`resultados/cache/train` y `val`), así que no se recalculan si la huella coincide. Los negativos se guardan aparte (`resultados/cache/neg_train` y `neg_val`) con el mismo preprocesamiento: ventana HU, Otsu + CLAHE y letterbox a 256 × 256. Construir los negativos de train es lo más lento y solo pasa una vez.

In [ ]:
def build_or_load_neg_cache(split, manifiesto, max_slices=None):
    # Igual que build_or_load_cache (Entrenamiento_multitarea), pero con los cortes del manifiesto de negativos.
    recs = manifiesto[manifiesto["split"] == split].sort_values(["case_id", "slice_index"])[["case_id", "slice_index"]]
    if max_slices:
        recs = recs.head(max_slices)
    tag = f"neg_{split}" + (f"_first{max_slices}" if max_slices else "") + ("_25d" if IN_CHANNELS == 3 else "")
    folder = CACHE_DIR / tag
    key = {**DATA_VERSION, "negativos": hashlib.sha1(recs.to_csv(index=False).encode()).hexdigest()[:12],
           "max_slices": max_slices}
    key_path = folder / "key.json"
    if key_path.exists() and json.loads(key_path.read_text()) == key:
        return _read_cache(tag, folder)

    folder.mkdir(parents=True, exist_ok=True)
    ds = LeanPengwinDetectionDataset(split=split, image_size=IMAGE_SIZE, channels=1, as_torch=False,
                                     slices_25d=IN_CHANNELS == 3)      # misma regla de vecinos que el caché de positivos
    ds.records = [{"case_id": str(c), "slice_index": int(s)} for c, s in zip(recs["case_id"], recs["slice_index"])]
    n, S = len(ds.records), IMAGE_SIZE
    images = np.lib.format.open_memmap(folder / "images.npy", mode="w+", dtype=np.float16, shape=(n, IN_CHANNELS, S, S))
    inst = np.lib.format.open_memmap(folder / "inst.npy", mode="w+", dtype=np.uint8, shape=(n, S, S))
    boxes_list, labels_list, case_ids, slice_ids = [], [], [], []
    t0 = time.perf_counter()
    for k in range(n):
        image, target = ds[k]
        mask = np.asarray(target["instance_mask"])
        assert np.asarray(image).shape == (IN_CHANNELS, S, S), np.asarray(image).shape
        assert len(target["boxes"]) == 0 and mask.max() == 0, f"el corte negativo {ds.records[k]} tiene pelvis"
        images[k] = np.asarray(image, dtype=np.float32).astype(np.float16)
        inst[k] = mask.astype(np.uint8)
        boxes_list.append(np.zeros((0, 4), np.float32)); labels_list.append(np.zeros(0, np.int64))
        case_ids.append(str(target["case_id"])); slice_ids.append(int(target["slice_index"]))
        if (k + 1) % 1000 == 0 or k + 1 == n:
            print(f"[{tag}] {k + 1}/{n} cortes | {time.perf_counter() - t0:.0f} s")
    boxes, labels = pad_boxes(boxes_list, labels_list)
    images.flush(); inst.flush()
    del images, inst, ds
    gc.collect()
    np.save(folder / "boxes.npy", boxes); np.save(folder / "labels.npy", labels)
    pd.DataFrame({"case_id": case_ids, "slice_index": slice_ids}).to_csv(folder / "meta.csv", index=False)
    key_path.write_text(json.dumps(key))          # al final: una construcción interrumpida no queda como válida
    print(f"[{tag}] caché creada: {n} cortes ({folder})")
    return _read_cache(tag, folder, quiet=True)


train_cache = build_or_load_cache("train", MAX_TRAIN_SLICES)
val_cache = build_or_load_cache("val", MAX_VAL_SLICES)
neg_train_cache = build_or_load_neg_cache("train", negativos, MAX_NEG_SLICES)
neg_val_cache = build_or_load_neg_cache("val", negativos, MAX_NEG_SLICES)

casos = {k: set(c["meta"]["case_id"]) for k, c in
         {"train": train_cache, "val": val_cache, "neg_train": neg_train_cache, "neg_val": neg_val_cache}.items()}
assert not ((casos["train"] | casos["neg_train"]) & (casos["val"] | casos["neg_val"])), "hay casos de val en train"
assert (neg_train_cache["labels"] == 0).all() and (neg_val_cache["labels"] == 0).all(), "un negativo trae cajas"
print({k: len(c["images"]) for k, c in {"train": train_cache, "val": val_cache, "neg_train": neg_train_cache,
                                         "neg_val": neg_val_cache}.items()})

## 5. Datos de entrenamiento y validación

- **Entrenamiento:** los positivos de siempre más **todos** los negativos de train, mezclados en cada época y en las tres etapas. En la etapa A un negativo enseña a la cabeza de detección a no poner cajas; en las etapas B y C enseña al decoder a no dibujar hueso cuando no lo hay.
- **Validación de detección y segmentación:** solo los cortes positivos de `val`, igual que en el entrenamiento original.
- **Validación de cortes vacíos:** el mejor score después del NMS en todos los cortes de `val`, positivos y negativos.

In [ ]:
class MezclaCortes(torch.utils.data.Dataset):
    # Positivos + un subconjunto de negativos. `need_seg` se propaga a los dos (train_stage lo cambia por etapa).
    def __init__(self, positivos, negativos, indices_negativos):
        self.pos, self.neg = positivos, negativos
        self.idx_neg = [int(i) for i in indices_negativos]

    @property
    def need_seg(self):
        return self.pos.need_seg

    @need_seg.setter
    def need_seg(self, value):
        self.pos.need_seg = self.neg.need_seg = bool(value)

    def __len__(self):
        return len(self.pos) + len(self.idx_neg)

    def __getitem__(self, i):
        return self.pos[i] if i < len(self.pos) else self.neg[self.idx_neg[i - len(self.pos)]]


INDICE_NEG_TRAIN = {(str(c), int(s)): k for k, (c, s) in
                    enumerate(zip(neg_train_cache["meta"]["case_id"], neg_train_cache["meta"]["slice_index"]))}
negativos_train_en_cache = negativos[[(str(c), int(s)) in INDICE_NEG_TRAIN
                                      for c, s in zip(negativos["case_id"], negativos["slice_index"])]]


def loader_entrenamiento(fraccion):
    sel = cv.submuestrear_negativos(negativos_train_en_cache, fraccion, seed=SEED)
    idx = [INDICE_NEG_TRAIN[(str(c), int(s))] for c, s in zip(sel["case_id"], sel["slice_index"])]
    ds = MezclaCortes(CachedSliceDataset(train_cache, CENTER_MODE), CachedSliceDataset(neg_train_cache, CENTER_MODE), idx)
    gen = torch.Generator(); gen.manual_seed(SEED)
    loader = torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS,
                                         pin_memory=DEVICE.type == "cuda", collate_fn=collate_slices, generator=gen)
    return loader, len(idx)


val_loader = torch.utils.data.DataLoader(CachedSliceDataset(val_cache, CENTER_MODE), batch_size=VAL_BATCH_SIZE, shuffle=False,
                                         num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda", collate_fn=collate_slices)

# Comprobación: un batch mezclado con negativos tiene la forma esperada y sus negativos no traen cajas ni máscara
_l, _n = loader_entrenamiento(1.0)
_ds = _l.dataset
_item_neg = _ds[len(_ds.pos)]
assert len(_item_neg["boxes"]) == 0 and int(_item_neg["instance_mask"].max()) == 0 and int(_item_neg["semantic"].max()) == 0
assert float(_item_neg["centers"].max()) == 0.0 and not bool(_item_neg["foreground"].any())
_b = collate_slices([_ds[0], _item_neg])
print(f"train: {len(_ds.pos)} positivos + {_n} negativos | batch: imagen {tuple(_b['image'].shape)}, "
      f"semántica {tuple(_b['semantic'].shape)} | val positivos: {len(val_loader.dataset)} | val negativos: {len(neg_val_cache['images'])}")
del _l, _ds, _item_neg, _b

## 6. Métricas de cortes vacíos y modelos de referencia

`evaluate` (del entrenamiento original) se envuelve para que, además de la mAP y el Dice, devuelva las métricas de cortes vacíos. La selección de la mejor época de cada etapa **no cambia** (`det`, `seg_gt`, `both`, como en el original), para que la única diferencia con el entrenamiento original sean los datos.

El modelo original y el ajustado se miden aquí con este mismo código, así la comparación de la sección 9 es justa.

In [ ]:
def cargar_exportado(ruta):
    modelo, payload = load_trained_model(ruta)
    arq = payload["architecture"]
    assert (arq["backbone_name"], arq["use_cbam"], arq.get("use_gate", True), arq.get("use_skips", True)) == \
           (BACKBONE_NAME, USE_CBAM, USE_CBAM_GATE, USE_SKIPS), f"{ruta.name}: arquitectura distinta {arq}"
    assert in_channels_del_modelo(payload) == IN_CHANNELS, f"{ruta.name}: {in_channels_del_modelo(payload)} canales, se esperaban {IN_CHANNELS}"
    return modelo, payload


@torch.no_grad()
def scores_cache(model, cache, batch_size=64):
    # Mejor score después del NMS en cada corte del caché (0.0 si no queda ninguna caja). Misma regla que scores_por_corte.
    model.eval()
    out = []
    for i in range(0, len(cache["images"]), batch_size):
        x = torch.from_numpy(cache["images"][i:i + batch_size].astype(np.float32)).to(DEVICE)
        with autocast():
            det_logits = model.det_head(model.trunk(x)[-1])
        final = postprocess(det_logits.float(), score_thresh=CONFIG_PIPELINE["score_thresh"], nms_fn=nms_por_clase,
                            iou_thresh=CONFIG_PIPELINE["iou_thresh"], max_det_per_class=CONFIG_PIPELINE["max_det_per_class"])
        out += [float(f["scores"].max()) if f["scores"].numel() else 0.0 for f in final]
    return np.asarray(out, dtype=np.float64)


evaluate_base = evaluate


def evaluate(model, loader, run_seg, instance_n=INSTANCE_EVAL_N):
    with torch.no_grad():                    # la versión original lleva @torch.no_grad(); se asegura aquí también
        out = evaluate_base(model, loader, run_seg, instance_n)
    s_pos, s_neg = scores_cache(model, val_cache), scores_cache(model, neg_val_cache)
    tiene = np.r_[np.ones(len(s_pos), bool), np.zeros(len(s_neg), bool)]
    r = cv.resumen_cortes_vacios(np.r_[s_pos, s_neg], tiene, recall_min=RECALL_MIN, umbral_fijo=UMBRAL_ACTUAL)
    out.update({f"val_{k}": v for k, v in r.items()})
    return out


COLUMNAS_CLAVE = ["val_det_map50", "val_det_mean_iou", "val_seg_dice_pred", "val_inst_recall50_pred",
                  "val_umbral_cal", "val_recall_pelvis_cal", "val_fpr_vacios_cal", "val_fpr_vacios_fijo", "val_vacios_con_caja"]

REFERENCIAS = {}
for nombre, ruta in {"original": MODELO_ORIGINAL, "ajustado": MODELO_AJUSTADO}.items():
    t0 = time.time()
    modelo, payload = cargar_exportado(ruta)
    REFERENCIAS[nombre] = evaluate(modelo, val_loader, run_seg=True)
    print(f"{nombre}: medido en {time.time() - t0:.0f} s")
    if not SMOKE_TEST:
        guardado = payload["selected_from"]["val_metrics"]
        for k in ("val_det_map50", "val_seg_dice_pred"):
            d = abs(REFERENCIAS[nombre][k] - guardado[k])
            print(f"   {k}: ahora {REFERENCIAS[nombre][k]:.4f} | en el checkpoint {guardado[k]:.4f} | diferencia {d:.4f}")
            if d > 0.005:
                print("   AVISO: no se reproduce el checkpoint. Revisar datos, caché o dispositivo antes de entrenar.")
    del modelo
    gc.collect()

LINEA_BASE = REFERENCIAS["original"]
(OUT_ROOT / "referencias_val.json").write_text(
    json.dumps({n: {k: float(v) for k, v in r.items()} for n, r in REFERENCIAS.items()}, indent=2), encoding="utf-8")
display(pd.DataFrame({n: {k: r[k] for k in COLUMNAS_CLAVE} for n, r in REFERENCIAS.items()}).round(4))

## 7. Entrenamiento: etapas A, B y C con negativos

Se crea un modelo nuevo con la misma semilla que el original y se recorren las tres etapas con `train_stage`, el mismo bucle del entrenamiento original. Cada etapa parte del mejor checkpoint de la anterior. Por época se guardan el historial (con las métricas de cortes vacíos), el mejor checkpoint y el último.

In [ ]:
train_loader, n_neg = loader_entrenamiento(FRACCION_NEGATIVOS)
try:
    commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=ROOT, timeout=10).stdout.strip() or None
except Exception:
    commit = None
CONFIG = {**CONFIG_ORIGINAL, "STAGES": STAGES, "USE_AMP": USE_AMP, "SMOKE_TEST": SMOKE_TEST, "BATCH_SIZE": BATCH_SIZE,
          "VAL_BATCH_SIZE": VAL_BATCH_SIZE, "INSTANCE_EVAL_N": INSTANCE_EVAL_N, "MAX_TRAIN_SLICES": MAX_TRAIN_SLICES,
          "MAX_VAL_SLICES": MAX_VAL_SLICES, "DATA_VERSION": DATA_VERSION, "torch": torch.__version__,
          "device": NOMBRE_DISPOSITIVO, "git_commit": commit,
          "NEGATIVOS_DESDE_CERO": {"fraccion_negativos": FRACCION_NEGATIVOS, "n_negativos_train": n_neg,
                                   "n_positivos_train": len(train_loader.dataset.pos),
                                   "referencias": {n: {k: float(r[k]) for k in COLUMNAS_CLAVE} for n, r in REFERENCIAS.items()}}}
(OUT_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2, default=str), encoding="utf-8")
print(f"Entrenamiento: {CONFIG['NEGATIVOS_DESDE_CERO']['n_positivos_train']} positivos + {n_neg} negativos "
      f"| {len(train_loader)} iteraciones por época")

seed_everything(SEED)
modelo = MultiTaskNet(BACKBONE_NAME, USE_CBAM, USE_CBAM_GATE, USE_SKIPS, IN_CHANNELS).to(DEVICE)
filas, t0 = [], time.time()
for etapa in STAGES:
    filas += train_stage(modelo, etapa, train_loader, val_loader, CONFIG)
historial = pd.DataFrame(filas)
historial.to_csv(OUT_DIR / "historial_completo.csv", index=False)
print(f"\nEntrenamiento completo en {(time.time() - t0) / 60:.1f} min | checkpoints: {sorted(p.name for p in MODEL_DIR.glob('*.pth'))}")
del modelo
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 8. Exportar el modelo entrenado desde cero

Se exporta el mejor checkpoint de la etapa C con el mismo formato que `modelo_multitarea_final.pt` y se comprueba que el archivo reconstruido da las mismas salidas. No reemplaza ningún modelo.

In [ ]:
ULTIMA = STAGES[-1]["name"]
CKPT_FINAL = MODEL_DIR / f"best_{ULTIMA}.pth"
EXPORT_PATH = MODEL_ROOT / "modelo_multitarea_negativos_completo.pt"
payload = export_inference_checkpoint(CKPT_FINAL, EXPORT_PATH)
payload["negativos_desde_cero"] = json.loads(json.dumps(CONFIG["NEGATIVOS_DESDE_CERO"], default=float))
torch.save(payload, EXPORT_PATH)

recargado, _ = cargar_exportado(EXPORT_PATH)
referencia = MultiTaskNet(BACKBONE_NAME, USE_CBAM, USE_CBAM_GATE, USE_SKIPS, IN_CHANNELS).to(DEVICE)
load_weights(referencia, CKPT_FINAL)
referencia.eval()
x = next(iter(val_loader))["image"][:4].to(DEVICE)
with torch.no_grad():
    assert torch.allclose(referencia.det_head(referencia.trunk(x)[-1]), recargado.det_head(recargado.trunk(x)[-1]), atol=1e-5)
print(f"Exportado {EXPORT_PATH.relative_to(ROOT)} ({EXPORT_PATH.stat().st_size / 1e6:.1f} MB), "
      f"etapa {payload['selected_from']['stage']} época {payload['selected_from']['epoch']}: reconstrucción OK")
del recargado, referencia
gc.collect()

## 9. Condición 1 de la regla: topes en 2D

Se compara la mejor época de la etapa C contra el original y el ajustado, con el mismo código. Un candidato pasa la condición 1 si no baja más de 0,01 en mAP@0.50 ni en Dice frente al original y conserva al menos el 99 % de los cortes con pelvis (con el umbral recalibrado). Las condiciones 2 y 3 se revisan en `salida_3d.ipynb`.

In [ ]:
mejor_c = historial[(historial["stage"] == ULTIMA) & (historial["is_best"] == True)].iloc[-1].to_dict()
CANDIDATOS_2D = {"ajustado": REFERENCIAS["ajustado"], "completo": mejor_c}

filas = [{"modelo": "original (referencia)", **{k: LINEA_BASE[k] for k in COLUMNAS_CLAVE}, "pasa_topes": None}]
topes = {}
for nombre, m in CANDIDATOS_2D.items():
    pasa, _, detalle = cv.criterio_seleccion(m, LINEA_BASE)
    topes[nombre] = {"pasa_topes": bool(pasa), **{k: float(v) for k, v in detalle.items()}}
    filas.append({"modelo": nombre, **{k: m[k] for k in COLUMNAS_CLAVE}, "pasa_topes": bool(pasa)})
comparacion_2d = pd.DataFrame(filas)
comparacion_2d.to_csv(OUT_ROOT / "comparacion_2d_val.csv", index=False)
(OUT_ROOT / "condicion1_topes.json").write_text(json.dumps({"smoke_test": SMOKE_TEST, "topes": topes,
                                                            "criterio": cv.CRITERIO}, indent=2), encoding="utf-8")
display(comparacion_2d.round(4))
for nombre, t in topes.items():
    print(f"{nombre}: {'pasa' if t['pasa_topes'] else 'NO pasa'} los topes | caída de mAP {t['caida_map50']:+.4f} | "
          f"caída de Dice {t['caida_dice']:+.4f} | recall de pelvis {t['recall_pelvis']:.3f} | cortes vacíos que pasan {t['fpr']:.1%}")

# Curvas por época de las tres etapas, frente al original y al ajustado
h = historial.reset_index(drop=True)
x = np.arange(1, len(h) + 1)
fig, ax = plt.subplots(1, 3, figsize=(16, 3.9), constrained_layout=True)
for a, k, t in zip(ax, ("val_fpr_vacios_cal", "val_det_map50", "val_seg_dice_pred"),
                   ("Cortes vacíos que pasan (umbral recalibrado)", "mAP@0.50 (cortes positivos)", "Dice con cajas predichas")):
    a.plot(x, h[k], marker="o", ms=3, label="desde cero con negativos")
    a.axhline(REFERENCIAS["original"][k], ls="--", c="gray", lw=1, label="original")
    a.axhline(REFERENCIAS["ajustado"][k], ls=":", c="k", lw=1, label="ajustado")
    for e in np.cumsum([s["epochs"] for s in STAGES])[:-1]:
        a.axvline(e + 0.5, c="gray", lw=0.6, alpha=0.4)
    a.set_title(t); a.set_xlabel("época (A | B | C)"); a.legend(fontsize=8); a.grid(alpha=0.25)
fig.savefig(FIG_DIR / "curvas_entrenamiento_negativos_completo.png", dpi=150)
plt.show()

## 10. Comprobación sobre los volúmenes completos de `val`

Igual que en `nms_pertenencia_pipeline` (§5.1): cada volumen de `val` se lee del MHA, se preprocesa corte a corte con sus vecinos z-1 y z+1 y se pasa por `scores_por_corte`. Se hace con los tres modelos. El original debe reproducir `scores_cortes_val.csv`, lo que confirma que la comparación es justa.

In [ ]:
def preprocesar_corte(corte_hu, output_size=IMAGE_SIZE):
    # Mismo preprocesamiento del dataset: ventaneo HU -> Otsu + CLAHE -> letterbox (igual que nms_pertenencia_pipeline).
    imagen = enhance_bone_contrast(apply_hu_window(corte_hu))
    imagen_256, _, transform = letterbox_pair(imagen, np.zeros(corte_hu.shape, dtype=np.int16), output_size)
    return imagen_256, transform


def scores_volumenes(model, casos_val):
    model.eval()
    comp, filas = {"backbone": model.backbone, "cbam": model.cbam, "det_head": model.det_head,
                   "features_fn": backbone_feature_pyramid}, []
    for caso in casos_val:
        fila = df[df["case_id"] == caso].iloc[0]
        vol = sitk.GetArrayFromImage(sitk.ReadImage(str(resolve_repo_path(fila["image_path"])))).astype(np.float32)
        lab = sitk.GetArrayFromImage(sitk.ReadImage(str(resolve_repo_path(fila["label_path"]))))
        tiene = lab.reshape(lab.shape[0], -1).any(axis=1)
        for inicio in range(0, vol.shape[0], 16):
            zs = range(inicio, min(inicio + 16, vol.shape[0]))
            x = torch.from_numpy(apilar_cortes(vol, zs, lambda corte: preprocesar_corte(corte)[0], IN_CHANNELS)).to(DEVICE)
            for z, s in zip(zs, scores_por_corte(comp, x)):
                filas.append({"case_id": caso, "z": z, "tiene_hueso": bool(tiene[z]), "score_max": float(s)})
        del vol, lab
        gc.collect()
    return pd.DataFrame(filas)


CASOS_VAL = sorted(df.loc[df["split"] == "val", "case_id"])[: (1 if SMOKE_TEST else None)]
volumenes = {}
for etiqueta, ruta in {"original": MODELO_ORIGINAL, "ajustado": MODELO_AJUSTADO, "completo": EXPORT_PATH}.items():
    modelo, _ = cargar_exportado(ruta)
    t0 = time.time()
    with torch.no_grad():
        tabla = scores_volumenes(modelo, CASOS_VAL)
    tabla.to_csv(OUT_ROOT / f"scores_cortes_val_{etiqueta}.csv", index=False)
    volumenes[etiqueta] = tabla
    print(f"{etiqueta}: {len(tabla)} cortes en {time.time() - t0:.0f} s")
    del modelo
    gc.collect()

guardados = pd.read_csv(METRICAS_DIR / "nms_pertenencia" / "scores_cortes_val.csv", dtype={"case_id": str})
_cmp = volumenes["original"].merge(guardados, on=["case_id", "z"], suffixes=("", "_guardado"))
print(f"Reproducción de scores_cortes_val.csv con el original: {len(_cmp)} cortes | diferencia máxima "
      f"{(_cmp['score_max'] - _cmp['score_max_guardado']).abs().max():.4f} | mismo tiene_hueso: "
      f"{bool((_cmp['tiene_hueso'] == _cmp['tiene_hueso_guardado']).all())}")

resumen_vol = pd.DataFrame({etq: cv.resumen_cortes_vacios(t["score_max"], t["tiene_hueso"], recall_min=RECALL_MIN,
                                                          umbral_fijo=UMBRAL_ACTUAL) for etq, t in volumenes.items()}).T
resumen_vol["vacios_que_pasan_cal"] = (resumen_vol["fpr_vacios_cal"] * resumen_vol["n_vacios"]).round().astype(int)
resumen_vol.to_csv(OUT_ROOT / "resumen_volumen_completo_val.csv")
display(resumen_vol.round(4))

fig, ax = plt.subplots(1, len(volumenes), figsize=(6 * len(volumenes), 3.8), constrained_layout=True, squeeze=False)
bins = np.linspace(0, 1, 41)
for a, (etq, t) in zip(ax[0], volumenes.items()):
    u = resumen_vol.loc[etq, "umbral_cal"]
    a.hist(t.loc[t["tiene_hueso"], "score_max"], bins=bins, alpha=0.7, label="con pelvis")
    a.hist(t.loc[~t["tiene_hueso"], "score_max"], bins=bins, alpha=0.7, label="sin pelvis")
    a.axvline(u, ls="--", c="k", lw=1, label=f"umbral recalibrado {u:.2f}")
    a.set_yscale("log"); a.set_xlabel("mejor score después del NMS"); a.set_title(etq); a.legend()
fig.savefig(FIG_DIR / "scores_cortes_val_tres_modelos.png", dpi=150)
plt.show()

## 11. Qué sigue

1. Subir a la rama: este notebook ejecutado, `resultados/metricas/entrenamiento_negativos_completo/`, `resultados/figuras/entrenamiento_negativos_completo/` y `resultados/modelos/entrenamiento_negativos_completo/modelo_multitarea_negativos_completo.pt` (no los `last_*.pth` ni las carpetas `_smoke`).
2. En `salida_3d.ipynb`, la celda de comparación final revisa las condiciones 2 y 3 de la regla (F1 3D con la salida reajustada para cada modelo y los 10 tramos sin pelvis con el control de "pelvis presente") y escribe la decisión.
3. Solo después se adopta el ganador y se vuelven a correr `nms_pertenencia_pipeline` y `salida_3d`.

**Limitaciones para el informe.** Una sola semilla: diferencias de uno o dos fragmentos pueden ser azar. La configuración 3D y la elección del modelo usan los mismos 5 casos de validación, así que los números de validación son optimistas; la medida definitiva es test, una sola vez.